# Chicago TNC : échantillons mensuels pour l’IA et références de pondération

## Ce que produit ce notebook
- Un CSV de trajets par mois, visant approximativement 2–3 Mo.
- Les effectifs réels par strate, les probabilités d’inclusion et poids d’expansion.
- Des indicateurs calculés sur l’ensemble des trajets publiés lus, un dictionnaire et un manifeste.

## Avant de lancer
Cette version parcourt **toutes les lignes du mois par API** pour réaliser un tirage uniforme reproductible dans chaque strate; elle ne prétend pas tirer un échantillon aléatoire côté serveur. Le volume réseau peut atteindre des centaines de Mo ou davantage et l’exécution peut être longue. Commencer avec janvier 2025, pas douze mois d’un coup. Aucune durée n’est garantie; respecter les limites du portail. Une panne interrompt le mois; les fichiers déjà terminés restent dans le dossier de sortie mais Colab peut effacer ce stockage à la fin d’une session.

1. Ouvrir dans Colab, lire les avertissements et les paramètres.
2. Régler `LANCER_PARCOURS_COMPLET = True`, puis Exécution → Tout exécuter.
3. À la fin, sauvegarder l’archive ZIP. Pour douze mois, modifier MOIS seulement après le pilote.
4. Un token Socrata est facultatif; le définir dans l’environnement, jamais dans un notebook publié.

Source : [Chicago TNP 2025 et suivantes](https://data.cityofchicago.org/Transportation/Transportation-Network-Providers-Trips-2025-/6dvr-xwnh). La version n’accepte pas 2024; cette période requiert une autre ressource et un audit de schéma. Le champ shared_trip_match existe dans le schéma 2025 vérifié. Horaires arrondis au quart d’heure et tarifs arrondis; données et géographies partiellement masquées.

## Périmètre de validation
Le format du notebook, la syntaxe et les fonctions sont testés sur données synthétiques. L’extraction réelle complète n’a pas été exécutée dans l’environnement de préparation. Des changements ou contraintes du serveur peuvent nécessiter un ajustement. Le notebook ne modifie ni GitHub, ni Notion, ni le backlog.


In [ ]:
# PILOTE : commencer par un seul mois. Ajouter d’autres mois après vérification.
MOIS = ['2025-01']  # Exemple annuel : [f'2025-{m:02d}' for m in range(1, 13)]
K_PAR_STRATE = 25  # Égalité de quotas; poids indispensables pour retrouver la demande réelle.
PAGE_SIZE = 50000
TIMEOUT = 180
PAUSE_SECONDES = 0.8
GRAINE = 20261101
DOSSIER_SORTIE = '/content/chicago_tnc_exports'  # Drive possible après montage explicite.
LANCER_PARCOURS_COMPLET = False  # Mettre True après lecture des avertissements.
# Token facultatif : variable d’environnement SOCRATA_APP_TOKEN. Ne jamais le publier.


In [ ]:
import io, json, time, hashlib, os
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from IPython.display import display

DOMAIN = 'https://data.cityofchicago.org'
DATASET = '6dvr-xwnh'
ENDPOINT = f'{DOMAIN}/resource/{DATASET}.json'
FIELDS = ['trip_id', 'trip_start_timestamp', 'trip_seconds', 'trip_miles',
          'pickup_community_area', 'dropoff_community_area', 'fare', 'trip_total',
          'shared_trip_authorized', 'shared_trip_match', 'trips_pooled']
NUMERIC = ['trip_seconds', 'trip_miles', 'fare', 'trip_total']
KEYS = ['jour_semaine', 'heure_depart', 'situation_partage']

session = requests.Session()
retry = Retry(total=5, backoff_factor=2, status_forcelist=[429, 500, 502, 503, 504], allowed_methods=['GET'], respect_retry_after_header=True)
session.mount('https://', HTTPAdapter(max_retries=retry))
session.headers['User-Agent'] = 'ChicagoTNC-Research-Notebook/1.0'
if os.getenv('SOCRATA_APP_TOKEN'):
    session.headers['X-App-Token'] = os.environ['SOCRATA_APP_TOKEN']

def get_json(url, params=None):
    response = session.get(url, params=params, timeout=TIMEOUT)
    response.raise_for_status()
    value = response.json()
    if isinstance(value, dict) and (value.get('error') or 'errorCode' in value):
        raise RuntimeError(str(value)[:600])
    return value

def query(**params):
    value = get_json(ENDPOINT, params)
    if not isinstance(value, list):
        raise RuntimeError('Réponse API inattendue : ' + str(value)[:400])
    return value

def metadata():
    return get_json(f'{DOMAIN}/api/views/{DATASET}.json')

def normaliser_booleen(series):
    return series.map(lambda x: str(x).strip().lower() if pd.notna(x) else '').map({'true': True, 'false': False}).astype('boolean')

def enrichir(raw):
    d = raw.reindex(columns=FIELDS).copy()
    date = pd.to_datetime(d.trip_start_timestamp, errors='coerce')
    if date.isna().any():
        raise ValueError('Date absente ou invalide : arrêter plutôt que perdre des lignes.')
    d['jour_semaine'] = date.dt.dayofweek.astype(int)
    d['heure_depart'] = date.dt.hour.astype(int)
    for field in NUMERIC + ['trips_pooled']:
        d[field] = pd.to_numeric(d[field], errors='coerce')
    for field in ['shared_trip_authorized', 'shared_trip_match']:
        d[field] = normaliser_booleen(d[field])
    auth, match = d.shared_trip_authorized, d.shared_trip_match
    d['situation_partage'] = 'inconnu'
    d.loc[(auth.eq(False) & match.eq(False)).fillna(False), 'situation_partage'] = 'non_autorise'
    d.loc[(auth.eq(True) & match.eq(False)).fillna(False), 'situation_partage'] = 'autorise_non_jumele'
    d.loc[(auth.eq(True) & match.eq(True)).fillna(False), 'situation_partage'] = 'jumele'
    d.loc[(auth.eq(False) & match.eq(True)).fillna(False), 'situation_partage'] = 'incoherent'
    d['_un'] = 1
    d['_auth_vrai'] = auth.eq(True).fillna(False).astype(int)
    d['_auth_connu'] = auth.notna().astype(int)
    d['_match_vrai'] = match.eq(True).fillna(False).astype(int)
    d['_match_connu'] = match.notna().astype(int)
    d['_match_autorise'] = (auth.eq(True) & match.eq(True)).fillna(False).astype(int)
    d['_match_connu_autorise'] = (auth.eq(True) & match.notna()).fillna(False).astype(int)
    return d

AGGS = {'N_population': ('_un', 'sum'),
        'N_autorise': ('_auth_vrai', 'sum'), 'N_autorisation_connue': ('_auth_connu', 'sum'),
        'N_jumele': ('_match_vrai', 'sum'), 'N_jumelage_connu': ('_match_connu', 'sum'),
        'N_jumele_autorise': ('_match_autorise', 'sum'),
        'N_jumelage_connu_autorise': ('_match_connu_autorise', 'sum')}
for field in NUMERIC:
    AGGS[f'{field}_somme'] = (field, 'sum')
    AGGS[f'{field}_valides'] = (field, 'count')

def reference_bloc(d):
    return d.groupby(KEYS, dropna=False).agg(**AGGS)

def reservoir_mettre_a_jour(reservoirs, d, rng, k):
    d = d.copy()
    d['_priorite'] = rng.random(len(d))
    for key, group in d.groupby(KEYS, dropna=False, sort=True):
        candidats = pd.concat([reservoirs[key], group], ignore_index=True) if key in reservoirs else group
        reservoirs[key] = candidats.nsmallest(k, '_priorite').copy()

def finaliser(reservoirs, references, mois):
    ref = pd.concat(references).groupby(level=KEYS).sum().reset_index()
    sample = pd.concat(list(reservoirs.values()), ignore_index=True)
    ns = sample.groupby(KEYS).size().rename('n_echantillon').reset_index()
    ref = ref.merge(ns, on=KEYS, how='left', validate='one_to_one')
    if ref.n_echantillon.isna().any() or (ref.n_echantillon <= 0).any():
        raise ValueError('Strate non vide sans observation dans l’échantillon')
    ref['probabilite_inclusion'] = ref.n_echantillon / ref.N_population
    ref['poids_expansion'] = ref.N_population / ref.n_echantillon
    sample = sample.merge(ref[KEYS + ['N_population', 'n_echantillon', 'probabilite_inclusion', 'poids_expansion']], on=KEYS, validate='many_to_one')
    sample['mois'] = mois
    ref['mois'] = mois
    sample['strate_id'] = sample[KEYS].astype(str).agg('_'.join, axis=1)
    ref['strate_id'] = ref[KEYS].astype(str).agg('_'.join, axis=1)
    sample = sample[['mois'] + FIELDS + KEYS + ['strate_id', 'N_population', 'n_echantillon', 'probabilite_inclusion', 'poids_expansion']]
    if sample.trip_id.duplicated().any():
        raise ValueError('Identifiants dupliqués dans l’échantillon')
    if not np.isclose(sample.poids_expansion.sum(), ref.N_population.sum()):
        raise ValueError('Somme des poids incorrecte')
    return sample.sort_values(KEYS + ['trip_id']), ref

def bilan_mensuel(ref, mois):
    totals = ref.select_dtypes(include='number').sum()
    result = {'mois': mois, 'N_population': int(totals.N_population), 'n_echantillon': int(totals.n_echantillon)}
    count_cols = [c for c in ref if c.startswith('N_')]
    for c in count_cols:
        result[c] = int(totals[c])
    def ratio(a, b):
        return float(totals[a] / totals[b]) if totals[b] else None
    result['taux_autorise_parmi_connus'] = ratio('N_autorise', 'N_autorisation_connue')
    result['taux_jumele_parmi_connus'] = ratio('N_jumele', 'N_jumelage_connu')
    result['taux_jumele_parmi_autorises_match_connu'] = ratio('N_jumele_autorise', 'N_jumelage_connu_autorise')
    for field in NUMERIC:
        result[f'{field}_somme'] = float(totals[f'{field}_somme'])
        result[f'{field}_valides'] = int(totals[f'{field}_valides'])
        result[f'{field}_moyenne'] = ratio(f'{field}_somme', f'{field}_valides')
    return result


In [ ]:
GUIDE_EXPORTS = '# Chicago TNC — échantillons pondérés\n\nChaque ligne est un trajet-client publié, pas un véhicule unique ni une personne unique. Chaque CSV mensuel conserve les variables sélectionnées, le mois, les clés de strate, l’effectif de population N, l’effectif échantillonné n, la probabilité n/N et le poids N/n. Jour de semaine : lundi=0, dimanche=6. Heure : 0–23 d’après l’horodatage local arrondi publié. Aucun passage en UTC ni correction de l’ambiguïté du changement d’heure.\n\nLes strates sont jour × heure × situation de partage : non_autorise (false,false), autorise_non_jumele (true,false), jumele (true,true), incoherent (false,true) et inconnu (au moins un indicateur absent ou illisible). Le jumelage utilise shared_trip_match, pas trips_pooled. trips_pooled ne décrit pas les passagers simultanément à bord et les courses d’un même ensemble partagé peuvent apparaître séparément.\n\nTirage : parcours exhaustif du mois par API, pagination par trip_id croissant. Attribuer à chaque ligne une priorité uniforme pseudo-aléatoire, conserver les k plus petites dans chaque strate; allocation fixe k par strate, avec recensement des strates de taille inférieure à k. Graine conservée. Aucun choix des premières lignes, aucun filtre sur la géographie, la distance ou la durée; les valeurs atypiques restent dans les données. Les champs numériques illisibles deviennent manquants; les analyses des valeurs valides doivent le signaler.\n\nLa cible 2–3 Mo est approximative : la taille dépend des strates et des valeurs. Aucune réduction automatique d’après le contenu de l’échantillon; modifier k puis refaire le tirage si nécessaire. Ne pas retirer arbitrairement des lignes sans recalculer le plan et les poids.\n\nRéférence : population_strates.csv contient les effectifs et sommes/comptes calculés sur toutes les lignes lues du mois; indicateurs_mensuels.csv fournit les totaux et ratios, indépendamment de l’échantillon. Les sommes d’un champ sans aucune valeur valide valent 0 par convention technique : lire les comptes de valeurs valides; ce zéro ne prouve pas une mesure nulle. Le dénominateur des taux de partage est explicitement limité aux indicateurs connus. Le taux parmi autorisés utilise les lignes autorisées avec shared_trip_match connu.\n\nEstimation pondérée d’une proportion : sum(poids * indicatrice) / sum(poids) sur le bon univers de valeurs connues. Moyenne d’une variable : sum(poids * valeur) / sum(poids) parmi valeurs valides, jamais moyenne brute des moyennes de strates. Total de trajets : sum(poids). Les poids ne sont pas une pondération résidentielle. Regrouper les mois avec les poids d’expansion conserve leurs différences de volume; ne pas normaliser chaque mois à un même total si l’on veut estimer un volume global. Les effectifs des strates de partage reproduisent par construction leurs totaux : ce n’est pas un test indépendant de représentativité. Les distributions fines, corridors rares, médianes et modèles nécessitent précautions, tailles suffisantes et variance tenant compte du plan. Aucun intervalle de confiance automatique n’est fourni.\n\nNe pas utiliser simultanément un total de référence et le même total estimé comme deux sources additionnables. Une course jumelée n’est pas une mesure des kilomètres mutualisés, des véhicules-kilomètres de flotte, du deadheading ou des GES.\n\nAPI : toutes les lignes sélectionnées sont transférées, même si seuls les échantillons sont conservés. Pagination monotone et contrôles avant/après limitent les risques de doublons et population mouvante, mais ne créent pas un instantané transactionnel. Un changement détecté arrête la validation du mois. Les mois terminés sont sauvegardés; aucune reprise automatique au milieu d’un mois. Le stockage Colab est temporaire : sauvegarder l’archive ou choisir un dossier Drive monté. Les comptes/tarifs sont ceux publiés, pas un registre exhaustif de tous les déplacements réels. Les géographies sont des zones, pas des positions individuelles exactes.\n\nSources : https://data.cityofchicago.org/Transportation/Transportation-Network-Providers-Trips-2025-/6dvr-xwnh ; dictionnaire récupéré à l’exécution via /api/views/6dvr-xwnh.json ; https://dev.socrata.com/consumers/getting-started.html . Vérifier licence, limites API et règles de confidentialité avant redistribution. Ne jamais publier un token.\n'

In [ ]:
if not LANCER_PARCOURS_COMPLET:
    raise RuntimeError('Lire les avertissements, puis mettre LANCER_PARCOURS_COMPLET = True et relancer. Le mois complet sera parcouru via API; cela peut durer longtemps.')
if not (1 <= int(K_PAR_STRATE) <= 500 and 1 <= int(PAGE_SIZE) <= 50000):
    raise ValueError('Vérifier K_PAR_STRATE et PAGE_SIZE')
if len(set(MOIS)) != len(MOIS):
    raise ValueError('MOIS contient des doublons')
for mois in MOIS:
    if len(mois) != 7 or pd.Period(mois, freq='M').strftime('%Y-%m') != mois or mois < '2025-01':
        raise ValueError('Cette version accepte seulement les mois 2025 et suivants au format AAAA-MM.')

out = Path(DOSSIER_SORTIE)
out.mkdir(parents=True, exist_ok=True)
run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
run_dir = out / ('execution_' + run_id)
run_dir.mkdir()
meta = metadata()
champs = {c.get('fieldName') for c in meta.get('columns', [])}
manquants = set(FIELDS) - champs
if manquants:
    raise ValueError('Schéma API incompatible : ' + str(manquants))
(run_dir / 'schema_source.json').write_text(json.dumps({'id': DATASET, 'name': meta.get('name'), 'columns': [{'fieldName': c.get('fieldName'), 'description': c.get('description'), 'dataTypeName': c.get('dataTypeName')} for c in meta.get('columns', []) if c.get('fieldName') in FIELDS]}, ensure_ascii=False, indent=2), encoding='utf-8')

references_tous, bilans, journals = [], [], []
for mois in MOIS:
    period = pd.Period(mois, freq='M')
    debut, fin = str(period.start_time.date()) + 'T00:00:00', str((period + 1).start_time.date()) + 'T00:00:00'
    where = f"trip_start_timestamp >= '{debut}' AND trip_start_timestamp < '{fin}'"
    before = metadata()
    n_attendu = int(query(**{'$select': 'count(*) AS n', '$where': where})[0]['n'])
    n_ids = int(query(**{'$select': 'count(trip_id) AS n', '$where': where})[0]['n'])
    if n_attendu == 0 or n_ids != n_attendu:
        raise ValueError(f'{mois}: population vide ou identifiants absents')
    print(f'{mois}: {n_attendu:,} trajets à parcourir. Sorties : {run_dir}')
    reservoirs, refs, dernier, total, pages = {}, [], None, 0, 0
    rng = np.random.default_rng(np.random.SeedSequence([GRAINE, period.year, period.month]))
    while True:
        curseur = " AND trip_id > '" + dernier.replace("'", "''") + "'" if dernier else ''
        rows = query(**{'$select': ','.join(FIELDS), '$where': where + curseur, '$order': 'trip_id ASC', '$limit': PAGE_SIZE})
        if not rows:
            break
        raw = pd.DataFrame(rows).reindex(columns=FIELDS)
        ids = raw.trip_id.astype(str)
        if ids.duplicated().any() or not ids.is_monotonic_increasing or (dernier is not None and ids.iloc[0] <= dernier):
            raise ValueError('Ordre ou unicité des identifiants invalide; extraction arrêtée')
        dernier = ids.iloc[-1]
        d = enrichir(raw)
        refs.append(reference_bloc(d))
        # Compacter les références régulièrement; ne pas garder tous les trajets.
        if len(refs) >= 20:
            refs = [pd.concat(refs).groupby(level=KEYS).sum()]
        reservoir_mettre_a_jour(reservoirs, d, rng, K_PAR_STRATE)
        total += len(d)
        pages += 1
        if total > n_attendu:
            raise RuntimeError('Population modifiée ou pagination incohérente')
        print(f'  {total:,}/{n_attendu:,} lignes; {pages} pages', end='\r')
        time.sleep(PAUSE_SECONDES)
    after = metadata()
    n_apres = int(query(**{'$select': 'count(*) AS n', '$where': where})[0]['n'])
    changements = [k for k in ['rowsUpdatedAt', 'dataUpdatedAt'] if before.get(k) is not None and before.get(k) != after.get(k)]
    if total != n_attendu or n_apres != n_attendu or changements:
        raise RuntimeError(f'{mois}: contrôle de stabilité échoué; ne pas considérer le mois comme complet. Relancer plus tard.')
    sample, ref = finaliser(reservoirs, refs, mois)
    if int(ref.N_population.sum()) != total:
        raise RuntimeError('Références incomplètes')
    filename = 'chicago_tnc_' + mois.replace('-', '_') + '.csv'
    path = run_dir / filename
    sample.to_csv(path, index=False, encoding='utf-8-sig', float_format='%.12g')
    ref.to_csv(run_dir / ('population_strates_' + mois + '.csv'), index=False, encoding='utf-8-sig', float_format='%.12g')
    bilan = bilan_mensuel(ref, mois)
    bilan['taille_echantillon_Mo'] = path.stat().st_size / 1000000
    bilan['pages_API'] = pages
    references_tous.append(ref)
    bilans.append(bilan)
    journals.append({'mois': mois, 'complet': True, 'lignes_lues': total, 'N_API_avant': n_attendu, 'N_API_apres': n_apres, 'rowsUpdatedAt': after.get('rowsUpdatedAt'), 'empreinte_csv_sha256': hashlib.sha256(path.read_bytes()).hexdigest(), 'dernier_trip_id': dernier})
    pd.concat(references_tous, ignore_index=True).to_csv(run_dir / 'population_strates.csv', index=False, encoding='utf-8-sig', float_format='%.12g')
    pd.DataFrame(bilans).to_csv(run_dir / 'indicateurs_mensuels.csv', index=False, encoding='utf-8-sig', float_format='%.12g')
    (run_dir / 'audit_mois.json').write_text(json.dumps(journals, ensure_ascii=False, indent=2), encoding='utf-8')
    print(f'\n{mois}: {len(sample):,} lignes, {bilan["taille_echantillon_Mo"]:.2f} Mo. Poids total = {sample.poids_expansion.sum():,.0f}')
    if not 2 <= bilan['taille_echantillon_Mo'] <= 3:
        print('Taille hors cible 2–3 Mo : ce n’est pas une erreur. Ajuster K_PAR_STRATE pour une future exécution; aucune troncature silencieuse.')
    del reservoirs, refs, raw, d, sample

manifest = {'execution_utc': run_id, 'dataset': DATASET, 'endpoint': ENDPOINT, 'mois': MOIS, 'k_par_strate': K_PAR_STRATE, 'graine': GRAINE, 'pagination': 'curseur trip_id croissant; contrôle effectifs avant/après', 'tirage': 'priorités uniformes pseudo-aléatoires indépendantes par ligne; k plus petites par strate', 'population': 'trajets publiés dont la date de départ arrondie appartient au mois', 'strates': KEYS, 'unites': {'trip_miles': 'miles', 'trip_seconds': 'secondes', 'fare': 'USD arrondi selon source', 'trip_total': 'USD, total publié avec arrondis'}, 'stabilite': 'Contrôles de version et d’effectifs; pas une transaction instantanée garantie du serveur.', 'limites': 'Pas de correction de non-publication ni de géographies masquées. Poids pour trajets, pas pour résidents. Pas d’intervalle de confiance automatique. Tarifs publiés arrondis. Aucun total de VKT de flotte ou PKT inféré.'}
(run_dir / 'manifeste.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
(run_dir / 'LISEZMOI.md').write_text(GUIDE_EXPORTS, encoding='utf-8')
# Contrôle des poids après sérialisation, avant livraison.
for mois in MOIS:
    s = pd.read_csv(run_dir / ('chicago_tnc_' + mois.replace('-', '_') + '.csv'))
    ref = pd.read_csv(run_dir / ('population_strates_' + mois + '.csv'))
    if not np.isclose(s.poids_expansion.sum(), ref.N_population.sum(), rtol=1e-9):
        raise ValueError('Contrôle des poids exportés échoué')
display(pd.DataFrame(bilans))
import zipfile
archive = out / ('chicago_tnc_' + run_id + '.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for path in run_dir.iterdir():
        if path.is_file():
            z.write(path, arcname=path.name)
try:
    from google.colab import files
    files.download(str(archive))
except ImportError:
    print('Archive :', archive.resolve())


## Vérification indicative des estimations
Une estimation sur échantillon peut différer de la référence. Ne pas exiger une égalité pour les distances ou tarifs; les seuls totaux des strates sont reproduits par construction.

In [ ]:
# Facultatif : après une extraction réussie, comparer moyenne de population et estimation.
if 'run_dir' in globals() and (run_dir / 'indicateurs_mensuels.csv').exists():
    mois_exemple = MOIS[0]
    s = pd.read_csv(run_dir / ('chicago_tnc_' + mois_exemple.replace('-', '_') + '.csv'))
    population = pd.read_csv(run_dir / 'indicateurs_mensuels.csv')
    valide = s.trip_miles.notna()
    moyenne_ponderee = np.average(s.loc[valide, 'trip_miles'], weights=s.loc[valide, 'poids_expansion'])
    print('Distance moyenne estimée sur échantillon pondéré, miles :', moyenne_ponderee)
    print('Distance moyenne de population, miles :', population.loc[population.mois.eq(mois_exemple), 'trip_miles_moyenne'].iloc[0])
    print('Une différence est normale : elle reflète notamment l’erreur d’échantillonnage.')
